# Import Libraries

In [2]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from statistics import mean
from rdkit.Chem import Descriptors, rdMolDescriptors, PandasTools, AllChem
from scipy.stats import loguniform, randint, uniform
from boruta import BorutaPy
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score, confusion_matrix,matthews_corrcoef
from sklearn.metrics import make_scorer, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, f1_score, roc_curve, precision_score, cohen_kappa_score, accuracy_score, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.model_selection import cross_validate
from sklearn.utils import resample
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Load your descriptor file here

In [3]:
data = pd.read_csv(r"Descriptors_External_Pubchem_with_activity.csv")

In [4]:
data

,PUBCHEM_CID,Smiles,IC50,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,156549456,CN(C)S(=O)(=O)NC(=O)C1=CN=C(O1)NC2=C3CCCC3=CC4...,50.0,12.126495,12.126495,0.160049,-3.885648,0.808368,15.666667,390.465,...,0,0,0,0,0,0,0,0,0,0
1,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,30.0,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,...,0,0,0,0,0,0,0,0,0,0
2,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,600.0,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,...,0,0,0,0,0,0,0,0,0,0
3,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,600.0,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,...,0,0,0,0,0,0,0,0,0,0
4,156549767,C1CCN(CC1)S(=O)(=O)NC(=O)C2=CN=C(O2)NC3=C4CCCC...,91.0,12.464700,12.464700,0.122864,-3.870787,0.755858,18.800000,430.530,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,170444535,COC(=O)C1COCCN1S(=O)(=O)NC(=O)C2=COC(=N2)NC3=C...,310.0,12.829858,12.829858,0.079721,-4.369221,0.541851,19.771429,524.983,...,0,0,0,0,0,0,0,0,0,0
421,156549744,CN(C)C1CCN(C1)S(=O)(=O)NC(=O)C2=COC(=N2)NC3=C4...,180.0,14.874356,14.874356,0.072550,-3.980880,0.656383,20.303030,477.562,...,0,0,0,0,0,0,0,0,0,0
422,156549741,CC1=C(C=C(C=C1)C(F)(F)F)NC2=NC(=CO2)C(=O)NS(=O...,32.0,12.961810,12.961810,0.038706,-4.564240,0.544796,16.428571,539.558,...,0,1,0,0,0,0,0,0,0,0
423,156549745,CC1=C(C=C(C=C1)C(F)(F)F)NC2=NC(=CO2)C(=O)NS(=O...,92.0,12.853326,12.853326,0.091038,-4.518154,0.773875,15.571429,418.397,...,0,0,0,0,0,0,0,0,0,0


In [5]:
eX_data =data.iloc[:, 3:]

In [6]:
eX_data

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,12.126495,12.126495,0.160049,-3.885648,0.808368,15.666667,390.465,368.289,390.136176,144,...,0,0,0,0,0,0,0,0,0,0
1,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,408.310,432.146741,160,...,0,0,0,0,0,0,0,0,0,0
2,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,408.310,432.146741,160,...,0,0,0,0,0,0,0,0,0,0
3,12.433450,12.433450,0.154114,-3.950787,0.739185,18.800000,432.502,408.310,432.146741,160,...,0,0,0,0,0,0,0,0,0,0
4,12.464700,12.464700,0.122864,-3.870787,0.755858,18.800000,430.530,404.322,430.167476,160,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,12.829858,12.829858,0.079721,-4.369221,0.541851,19.771429,524.983,499.783,524.113248,188,...,0,0,0,0,0,0,0,0,0,0
421,14.874356,14.874356,0.072550,-3.980880,0.656383,20.303030,477.562,449.338,477.184604,178,...,0,0,0,0,0,0,0,0,0,0
422,12.961810,12.961810,0.038706,-4.564240,0.544796,16.428571,539.558,515.366,539.112010,194,...,0,1,0,0,0,0,0,0,0,0
423,12.853326,12.853326,0.091038,-4.518154,0.773875,15.571429,418.397,401.261,418.092261,152,...,0,0,0,0,0,0,0,0,0,0


# Normilization of data

In [7]:
with open("standard_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
data_scaled = scaler.transform(eX_data)

In [8]:
list2=eX_data.columns
external1= pd.DataFrame(data_scaled,columns=list2)

# Selected fatures

In [9]:
with open("selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)
# Apply to new data
selected = external1[selected_features]

In [10]:
selected

,MinEStateIndex,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,BCUT2D_MWHI,BCUT2D_CHGLO,BCUT2D_LOGPLOW,BCUT2D_MRHI,Chi0,...,SMR_VSA5,SMR_VSA7,SMR_VSA9,SlogP_VSA6,TPSA,EState_VSA8,VSA_EState1,VSA_EState2,VSA_EState6,MolLogP
0,-0.796107,0.030458,0.023231,0.030265,0.106667,0.000690,0.167066,-0.326347,0.374637,0.038699,...,0.000000,-0.609797,0.0,-0.648477,0.610487,-0.021060,0.760971,-0.035605,-0.403887,-0.856478
1,-0.813592,0.400237,0.393868,0.400539,0.533333,0.000690,-0.896041,-0.847068,0.374910,0.352542,...,0.000000,-0.609797,0.0,-0.648477,0.889023,-0.021060,1.014565,-0.003545,-0.403212,-1.030527
2,-0.813592,0.400237,0.393868,0.400539,0.533333,0.000690,-0.896041,-0.847068,0.374910,0.352542,...,0.000000,-0.609797,0.0,-0.648477,0.889023,-0.021060,1.014565,-0.003545,-0.403212,-1.030527
3,-0.813592,0.400237,0.393868,0.400539,0.533333,0.000690,-0.896041,-0.847068,0.374910,0.352542,...,0.000000,-0.609797,0.0,-0.648477,0.889023,-0.021060,1.014565,-0.003545,-0.403212,-1.030527
4,-0.792118,0.382890,0.356935,0.383094,0.533333,0.000690,-0.876961,-0.690301,0.374901,0.352542,...,0.579732,-0.609797,0.0,-0.648477,0.610487,0.277749,0.834662,0.001088,-0.399825,-0.155202
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,-0.925911,1.213746,1.241005,1.211117,1.280000,0.207958,-1.562505,-1.348857,0.375186,1.175189,...,0.181838,-0.661428,0.0,-0.878337,1.682686,-0.105189,1.202318,1.102987,-0.829795,-0.882578
421,-0.821670,0.796607,0.773831,0.797496,1.013333,0.000690,-0.353618,-0.589550,0.374961,0.886142,...,0.375082,-0.622122,0.0,-0.878337,0.708261,-0.106124,1.394916,0.188457,-0.676694,-0.694570
422,-0.978259,1.341955,1.385320,1.343313,1.440000,0.000714,-1.532612,-0.946196,0.385415,1.469409,...,-0.220707,-0.560042,0.0,-0.181136,1.738514,-0.404932,3.200566,-0.038281,-0.382648,-0.747209
423,-0.965888,0.276162,0.328587,0.276665,0.320000,0.000690,0.173900,-0.489008,0.374938,0.313661,...,-0.378711,-0.560042,0.0,-0.181136,0.610487,-0.404932,2.210285,-0.064837,-0.363417,-0.185763


In [11]:
import pandas as pd
import pickle

# Load saved models
with open("rf_model.pkl", "rb") as f:
    mod_rf = pickle.load(f)

with open("adbt_model.pkl", "rb") as f:
    mod_adbt = pickle.load(f) 

# External feature set
df_predict_features = selected

# Predictions
df_predict_clf = pd.DataFrame()
df_predict_clf['Class_pred_RandomForest'] = mod_rf.predict(df_predict_features)
df_predict_clf['Class_pred_AdaBoost'] = mod_adbt.predict(df_predict_features)


# Metadata
t1 = pd.DataFrame(data['PUBCHEM_CID'].values, columns=['PUBCHEM_CID'])
t2 = pd.DataFrame(data['Smiles'].values, columns=['Smiles'])
t3 = pd.DataFrame(data['IC50'].values, columns=['IC50'])

# Final output
result = pd.concat([t1, t2, t3, df_predict_clf], axis=1)

In [12]:
result.to_csv('Predictions_External_Pubchem_with_activity.csv')

In [13]:
result

,PUBCHEM_CID,Smiles,IC50,Class_pred_RandomForest,Class_pred_AdaBoost
0,156549456,CN(C)S(=O)(=O)NC(=O)C1=CN=C(O1)NC2=C3CCCC3=CC4...,50.0,1,1
1,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,30.0,1,0
2,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,600.0,1,0
3,156549766,C1CC2=CC3=C(CCC3)C(=C2C1)NC4=NC=C(O4)C(=O)NS(=...,600.0,1,0
4,156549767,C1CCN(CC1)S(=O)(=O)NC(=O)C2=CN=C(O2)NC3=C4CCCC...,91.0,1,1
...,...,...,...,...,...
420,170444535,COC(=O)C1COCCN1S(=O)(=O)NC(=O)C2=COC(=N2)NC3=C...,310.0,1,0
421,156549744,CN(C)C1CCN(C1)S(=O)(=O)NC(=O)C2=COC(=N2)NC3=C4...,180.0,1,1
422,156549741,CC1=C(C=C(C=C1)C(F)(F)F)NC2=NC(=CO2)C(=O)NS(=O...,32.0,1,1
423,156549745,CC1=C(C=C(C=C1)C(F)(F)F)NC2=NC(=CO2)C(=O)NS(=O...,92.0,1,1
